In [9]:
import numpy as np
import sounddevice as sd
from scipy import signal
from scipy.io import wavfile

# For MP3 conversion, we'll use pydub which requires ffmpeg
try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False
    print("MP3 export requires pydub. Install with: pip install pydub")

def generate_meditation_sounds(inhale_time=4, exhale_time=4, duration=60, 
                              play_audio=True, save_wav=True, save_mp3=True,
                              output_filename="meditation"):
    """
    Generate calming inhale/exhale meditation sounds and save as WAV/MP3
    
    Parameters:
    - inhale_time: Duration of inhale sound in seconds
    - exhale_time: Duration of exhale sound in seconds
    - duration: Total duration of the meditation session in seconds
    - play_audio: Whether to play the audio while generating
    - save_wav: Whether to save as WAV file
    - save_mp3: Whether to save as MP3 file (requires pydub and ffmpeg)
    - output_filename: Base filename for saved audio (without extension)
    """
    # Sample rate
    sample_rate = 44100
    
    def generate_breathing_cycle():
        """Generate one full breathing cycle (inhale + exhale)"""
        # === INHALE SOUND ===
        inhale_samples = int(inhale_time * sample_rate)
        t_inhale = np.linspace(0, inhale_time, inhale_samples, False)
        
        # Create a rising amplitude modulation for inhale (starts soft, grows stronger)
        inhale_amplitude = np.linspace(0.3, 1.0, inhale_samples)
        
        # Create a rising frequency modulation for inhale
        # Inhale starts with lower frequencies and adds higher frequencies as it progresses
        inhale_base = 300  # Base frequency
        
        # Create several harmonic components
        inhale_sound = np.zeros(inhale_samples)
        
        # First harmonic - always present but swells
        harmonic1 = np.sin(2 * np.pi * inhale_base * t_inhale)
        harmonic1_env = np.linspace(0.5, 1.0, inhale_samples)
        inhale_sound += harmonic1 * harmonic1_env * 0.7
        
        # Second harmonic - grows in
        harmonic2 = np.sin(2 * np.pi * inhale_base * 1.5 * t_inhale)
        harmonic2_env = np.power(np.linspace(0, 1, inhale_samples), 2)  # Quadratic rise
        inhale_sound += harmonic2 * harmonic2_env * 0.4
        
        # Third harmonic - subtle high frequency "air" sound
        harmonic3 = np.sin(2 * np.pi * inhale_base * 2.5 * t_inhale)
        harmonic3_env = np.power(np.linspace(0, 1, inhale_samples), 3)  # Cubic rise
        inhale_sound += harmonic3 * harmonic3_env * 0.25
        
        # Add a subtle "rush of air" sound that increases during the inhale
        noise = np.random.normal(0, 0.1, inhale_samples)
        b_inhale, a_inhale = signal.butter(2, 0.05, 'highpass')
        filtered_noise = signal.lfilter(b_inhale, a_inhale, noise)
        noise_env = np.power(np.linspace(0, 1, inhale_samples), 2)
        inhale_sound += filtered_noise * noise_env * 0.5
        
        # Apply overall envelope 
        inhale_env = np.ones(inhale_samples)
        # Smooth attack
        attack_len = int(0.1 * inhale_samples)
        inhale_env[:attack_len] = np.linspace(0, 1, attack_len)
        # Apply overall amplitude progression - breath gets stronger as inhale progresses
        inhale_env = inhale_env * inhale_amplitude
        
        inhale_sound = inhale_sound * inhale_env * 0.7  # Scale to avoid clipping
        
        # === EXHALE SOUND ===
        exhale_samples = int(exhale_time * sample_rate)
        t_exhale = np.linspace(0, exhale_time, exhale_samples, False)
        
        # Create a falling amplitude modulation for exhale (starts strong, gets softer)
        exhale_amplitude = np.linspace(1.0, 0.2, exhale_samples)
        
        # Exhale has more low frequencies and is "airier"
        exhale_base = 220  # Lower base frequency than inhale
        
        # Create exhale sound
        exhale_sound = np.zeros(exhale_samples)
        
        # First harmonic - starts strong, fades
        harmonic1 = np.sin(2 * np.pi * exhale_base * t_exhale)
        harmonic1_env = np.linspace(1.0, 0.4, exhale_samples)
        exhale_sound += harmonic1 * harmonic1_env * 0.6
        
        # Second harmonic - fades faster
        harmonic2 = np.sin(2 * np.pi * exhale_base * 1.2 * t_exhale)
        harmonic2_env = np.power(np.linspace(1, 0, exhale_samples), 1.5)
        exhale_sound += harmonic2 * harmonic2_env * 0.35
        
        # Add a stronger "rush of air" sound that decreases during the exhale
        noise = np.random.normal(0, 0.15, exhale_samples)
        b_exhale, a_exhale = signal.butter(2, 0.07, 'lowpass')
        filtered_noise = signal.lfilter(b_exhale, a_exhale, noise)
        noise_env = np.power(np.linspace(1, 0, exhale_samples), 1.5)
        exhale_sound += filtered_noise * noise_env * 0.6
        
        # Apply overall envelope
        exhale_env = np.ones(exhale_samples)
        # Smooth release
        release_len = int(0.15 * exhale_samples)
        if release_len > 0:
            exhale_env[-release_len:] = np.linspace(1, 0, release_len)
        # Apply overall amplitude progression
        exhale_env = exhale_env * exhale_amplitude
        
        exhale_sound = exhale_sound * exhale_env * 0.8  # Scale to avoid clipping
        
        return inhale_sound, exhale_sound
    
    # Generate full session
    cycles = int(duration / (inhale_time + exhale_time))
    print(f"Generating meditation session for {duration} seconds ({cycles} cycles)")
    
    # Pre-allocate full audio array
    full_audio = np.array([], dtype=np.float32)
    
    # Generate each cycle
    for i in range(cycles):
        print(f"Generating cycle {i+1}/{cycles}")
        
        # Get inhale and exhale sounds for this cycle
        inhale_sound, exhale_sound = generate_breathing_cycle()
        
        # Play audio if requested
        if play_audio:
            print("Inhale...")
            sd.play(inhale_sound, sample_rate)
            sd.wait()
            
            print("Exhale...")
            sd.play(exhale_sound, sample_rate)
            sd.wait()
        
        # Append to full audio
        full_audio = np.append(full_audio, inhale_sound)
        full_audio = np.append(full_audio, exhale_sound)
    
    print(f"Generated audio length: {len(full_audio)} samples")
    
    # Save as WAV file
    if save_wav:
        wav_filename = f"{output_filename}.wav"
        print(f"Saving meditation to {wav_filename}...")
        
        # Normalize to avoid clipping
        max_amplitude = np.max(np.abs(full_audio))
        if max_amplitude > 0:
            full_audio = full_audio / max_amplitude * 0.95
            
        # Convert to 16-bit int format
        normalized_audio = np.int16(full_audio * 32767)
        wavfile.write(wav_filename, sample_rate, normalized_audio)
        print(f"WAV file saved with {len(normalized_audio)} samples")
    
    # Save as MP3 file
    if save_mp3 and PYDUB_AVAILABLE:
        mp3_filename = f"{output_filename}.mp3"
        print(f"Converting to MP3 format ({mp3_filename})...")
        
        # Create a temporary WAV for conversion
        temp_wav = "_temp_for_mp3_conversion.wav"
        # Normalize to avoid clipping
        max_amplitude = np.max(np.abs(full_audio))
        if max_amplitude > 0:
            full_audio = full_audio / max_amplitude * 0.95
        normalized_audio = np.int16(full_audio * 32767)
        wavfile.write(temp_wav, sample_rate, normalized_audio)
        
        # Convert WAV to MP3
        try:
            audio = AudioSegment.from_wav(temp_wav)
            audio.export(mp3_filename, format="mp3", bitrate="192k")
            print(f"MP3 file saved successfully")
            
            # Remove temp file
            import os
            if os.path.exists(temp_wav):
                os.remove(temp_wav)
        except Exception as e:
            print(f"Error converting to MP3: {e}")
    elif save_mp3 and not PYDUB_AVAILABLE:
        print("MP3 export requires pydub and ffmpeg.")
    
    print("Meditation audio generation complete.")
    return full_audio

# Run the script
if __name__ == "__main__":
    audio = generate_meditation_sounds(
        inhale_time=6, 
        exhale_time=6, 
        duration=12,  # 2 minutes
        play_audio=True,
        save_wav=True,
        save_mp3=False,
        output_filename="natural_breathing_meditation"
    )

Generating meditation session for 12 seconds (1 cycles)
Generating cycle 1/1
Inhale...
Exhale...
Generated audio length: 529200 samples
Saving meditation to natural_breathing_meditation.wav...
WAV file saved with 529200 samples
Meditation audio generation complete.


In [5]:
import numpy as np
import sounddevice as sd
from scipy import signal
from scipy.io import wavfile

# For MP3 conversion, we'll use pydub which requires ffmpeg
try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False
    print("MP3 export requires pydub. Install with: pip install pydub")

def generate_breathing_sounds(inhale_time=4, exhale_time=4, duration=60, 
                              play_audio=True, save_wav=True, save_mp3=True,
                              output_filename="meditation", save_separate=True):
    """
    Generate calming inhale/exhale meditation sounds and save as WAV/MP3
    
    Parameters:
    - inhale_time: Duration of inhale sound in seconds
    - exhale_time: Duration of exhale sound in seconds
    - duration: Total duration of the meditation session in seconds
    - play_audio: Whether to play the audio while generating
    - save_wav: Whether to save as WAV file
    - save_mp3: Whether to save as MP3 file (requires pydub and ffmpeg)
    - output_filename: Base filename for saved audio (without extension)
    - save_separate: Whether to save separate inhale.wav and exhale.wav files
    """
    # Sample rate
    sample_rate = 44100
    
    def generate_breathing_cycle():
        """Generate one full breathing cycle (inhale + exhale)"""
        # === INHALE SOUND ===
        inhale_samples = int(inhale_time * sample_rate)
        t_inhale = np.linspace(0, inhale_time, inhale_samples, False)
        
        # Create a rising amplitude modulation for inhale (starts soft, grows stronger)
        inhale_amplitude = np.linspace(0.01, 0.2, inhale_samples)
        
        # Create a rising frequency modulation for inhale
        # Inhale starts with lower frequencies and adds higher frequencies as it progresses
        inhale_base = 100  # Base frequency
        
        # Create several harmonic components
        inhale_sound = np.zeros(inhale_samples)
        
        # First harmonic - always present but swells
        harmonic1 = np.sin(2 * np.pi * inhale_base * t_inhale)
        harmonic1_env = np.linspace(0.5, 1.0, inhale_samples)
        inhale_sound += harmonic1 * harmonic1_env * 0.7
        
        # Second harmonic - grows in
        harmonic2 = np.sin(2 * np.pi * inhale_base * 1.5 * t_inhale)
        harmonic2_env = np.power(np.linspace(0, 1, inhale_samples), 2)  # Quadratic rise
        inhale_sound += harmonic2 * harmonic2_env * 0.4
        
        # Third harmonic - subtle high frequency "air" sound
        harmonic3 = np.sin(2 * np.pi * inhale_base * 2.5 * t_inhale)
        harmonic3_env = np.power(np.linspace(0, 1, inhale_samples), 3)  # Cubic rise
        inhale_sound += harmonic3 * harmonic3_env * 0.25
        
        # Add a subtle "rush of air" sound that increases during the inhale
        noise = np.random.normal(0, 0.1, inhale_samples)
        b_inhale, a_inhale = signal.butter(2, 0.05, 'highpass')
        filtered_noise = signal.lfilter(b_inhale, a_inhale, noise)
        noise_env = np.power(np.linspace(0, 1, inhale_samples), 2)
        inhale_sound += filtered_noise * noise_env * 0.5
        
        # Apply overall envelope 
        inhale_env = np.ones(inhale_samples)
        # Smooth attack
        attack_len = int(0.1 * inhale_samples)
        inhale_env[:attack_len] = np.linspace(0, 1, attack_len)
        # Apply overall amplitude progression - breath gets stronger as inhale progresses
        inhale_env = inhale_env * inhale_amplitude
        
        inhale_sound = inhale_sound * inhale_env * 0.5  # Scale to avoid clipping
        
        # === EXHALE SOUND ===
        exhale_samples = int(exhale_time * sample_rate)
        t_exhale = np.linspace(0, exhale_time, exhale_samples, False)
        
        # Create a falling amplitude modulation for exhale (starts strong, gets softer)
        exhale_amplitude = np.linspace(0.2, 0.01, exhale_samples)
        
        # Exhale has more low frequencies and is "airier"
        exhale_base = 70  # Lower base frequency than inhale
        
        # Create exhale sound
        exhale_sound = np.zeros(exhale_samples)
        
        # First harmonic - starts strong, fades
        harmonic1 = np.sin(2 * np.pi * exhale_base * t_exhale)
        harmonic1_env = np.linspace(1.0, 0.4, exhale_samples)
        exhale_sound += harmonic1 * harmonic1_env * 0.6
        
        # Second harmonic - fades faster
        harmonic2 = np.sin(2 * np.pi * exhale_base * 1.2 * t_exhale)
        harmonic2_env = np.power(np.linspace(1, 0, exhale_samples), 1.5)
        exhale_sound += harmonic2 * harmonic2_env * 0.35
        
        # Add a stronger "rush of air" sound that decreases during the exhale
        noise = np.random.normal(0, 0.15, exhale_samples)
        b_exhale, a_exhale = signal.butter(2, 0.07, 'lowpass')
        filtered_noise = signal.lfilter(b_exhale, a_exhale, noise)
        noise_env = np.power(np.linspace(1, 0, exhale_samples), 1.5)
        exhale_sound += filtered_noise * noise_env * 0.6
        
        # Apply overall envelope
        exhale_env = np.ones(exhale_samples)
        # Smooth release
        release_len = int(0.15 * exhale_samples)
        if release_len > 0:
            exhale_env[-release_len:] = np.linspace(1, 0, release_len)
        # Apply overall amplitude progression
        exhale_env = exhale_env * exhale_amplitude
        
        exhale_sound = exhale_sound * exhale_env * 0.5  # Scale to avoid clipping
        
        return inhale_sound, exhale_sound
    
    # First, generate a single inhale/exhale cycle and save them separately if requested
    example_inhale, example_exhale = generate_breathing_cycle()

    inhale_volume_factor = 10 ** (-300 / 20)
    exhale_volume_factor = 10 ** (-1000 / 20)
    example_inhale = example_inhale * inhale_volume_factor
    example_exhale = example_exhale * exhale_volume_factor
    
    # Save these as separate files if requested
    if save_separate and save_wav:
        # Save inhale sound
        inhale_filename = "inhale.wav"
        print(f"Saving single inhale sound to {inhale_filename}...")
        # Normalize to avoid clipping
        inhale_max = np.max(np.abs(example_inhale))
        if inhale_max > 0:
            normalized_inhale = example_inhale / inhale_max * 0.95
        else:
            normalized_inhale = example_inhale
        # Convert to 16-bit int format
        int16_inhale = np.int16(normalized_inhale * 32767)
        wavfile.write(inhale_filename, sample_rate, int16_inhale)
        
        # Save exhale sound
        exhale_filename = "exhale.wav"
        print(f"Saving single exhale sound to {exhale_filename}...")
        # Normalize to avoid clipping
        exhale_max = np.max(np.abs(example_exhale))
        if exhale_max > 0:
            normalized_exhale = example_exhale / exhale_max * 0.95
        else:
            normalized_exhale = example_exhale
        # Convert to 16-bit int format
        int16_exhale = np.int16(normalized_exhale * 32767)
        wavfile.write(exhale_filename, sample_rate, int16_exhale)
        
        print("Separate inhale and exhale files saved successfully")
    
    # Now generate the full meditation session
    cycles = int(duration / (inhale_time + exhale_time))
    print(f"Generating full meditation session for {duration} seconds ({cycles} cycles)")
    
    # Pre-allocate full audio array
    full_audio = np.array([], dtype=np.float32)
    
    # Generate each cycle
    for i in range(cycles):
        print(f"Generating cycle {i+1}/{cycles}")
        
        # Get inhale and exhale sounds for this cycle
        # Note: We regenerate each cycle for natural variations
        inhale_sound, exhale_sound = generate_breathing_cycle()
        
        # Play audio if requested
        if play_audio:
            print("Inhale...")
            sd.play(inhale_sound, sample_rate)
            sd.wait()
            
            print("Exhale...")
            sd.play(exhale_sound, sample_rate)
            sd.wait()
        
        # Append to full audio
        full_audio = np.append(full_audio, inhale_sound)
        full_audio = np.append(full_audio, exhale_sound)
    
    print(f"Generated full audio length: {len(full_audio)} samples")
    
    # Save full session as WAV file
    if save_wav:
        wav_filename = f"{output_filename}.wav"
        print(f"Saving full meditation to {wav_filename}...")
        
        # Normalize to avoid clipping
        max_amplitude = np.max(np.abs(full_audio))
        if max_amplitude > 0:
            full_audio = full_audio / max_amplitude * 0.95
            
        # Convert to 16-bit int format
        normalized_audio = np.int16(full_audio * 32767)
        wavfile.write(wav_filename, sample_rate, normalized_audio)
        print(f"Full WAV file saved with {len(normalized_audio)} samples")
    
    # Save full session as MP3 file
    if save_mp3 and PYDUB_AVAILABLE:
        mp3_filename = f"{output_filename}.mp3"
        print(f"Converting full meditation to MP3 format ({mp3_filename})...")
        
        # Create a temporary WAV for conversion
        temp_wav = "_temp_for_mp3_conversion.wav"
        # Normalize to avoid clipping
        max_amplitude = np.max(np.abs(full_audio))
        if max_amplitude > 0:
            full_audio = full_audio / max_amplitude * 0.95
        normalized_audio = np.int16(full_audio * 32767)
        wavfile.write(temp_wav, sample_rate, normalized_audio)
        
        # Convert WAV to MP3
        try:
            audio = AudioSegment.from_wav(temp_wav)
            audio.export(mp3_filename, format="mp3", bitrate="192k")
            print(f"MP3 file saved successfully")
            
            # Remove temp file
            import os
            if os.path.exists(temp_wav):
                os.remove(temp_wav)
        except Exception as e:
            print(f"Error converting to MP3: {e}")
    elif save_mp3 and not PYDUB_AVAILABLE:
        print("MP3 export requires pydub and ffmpeg.")
    
    # Convert separate inhale/exhale to MP3 if requested
    if save_separate and save_mp3 and PYDUB_AVAILABLE:
        try:
            # Convert inhale to MP3
            inhale_mp3 = "inhale.mp3"
            inhale_audio = AudioSegment.from_wav("inhale.wav")
            inhale_audio.export(inhale_mp3, format="mp3", bitrate="192k")
            
            # Convert exhale to MP3
            exhale_mp3 = "exhale.mp3"
            exhale_audio = AudioSegment.from_wav("exhale.wav")
            exhale_audio.export(exhale_mp3, format="mp3", bitrate="192k")
            
            print("Separate inhale and exhale MP3 files saved successfully")
        except Exception as e:
            print(f"Error converting separate files to MP3: {e}")
    
    print("Meditation audio generation complete.")
    return full_audio, example_inhale, example_exhale

# Run the script
if __name__ == "__main__":
    full_audio, inhale, exhale = generate_breathing_sounds(
        inhale_time=16, 
        exhale_time=16, 
        duration=32,  # 2 minutes
        play_audio=True,
        save_wav=True,
        save_mp3=False,
        output_filename="natural_breathing_meditation",
        save_separate=True  # Save separate inhale.wav and exhale.wav
    )

Saving single inhale sound to inhale.wav...
Saving single exhale sound to exhale.wav...
Separate inhale and exhale files saved successfully
Generating full meditation session for 32 seconds (1 cycles)
Generating cycle 1/1
Inhale...
Exhale...
Generated full audio length: 1411200 samples
Saving full meditation to natural_breathing_meditation.wav...
Full WAV file saved with 1411200 samples
Meditation audio generation complete.


In [8]:
import numpy as np
import sounddevice as sd
from scipy import signal
from scipy.io import wavfile

# For MP3 conversion
try:
    from pydub import AudioSegment
    PYDUB_AVAILABLE = True
except ImportError:
    PYDUB_AVAILABLE = False
    print("MP3 export requires pydub. Install with: pip install pydub")

def generate_meditation_sounds(inhale_time=4, exhale_time=4, duration=60, 
                              play_audio=True, save_wav=True, save_mp3=True,
                              output_filename="meditation"):
    """
    Generate realistic human-like breathing sounds for meditation
    """
    # Sample rate
    sample_rate = 44100
    
    def generate_human_breath_cycle():
        """Generate realistic human inhale and exhale sounds"""
        inhale_samples = int(inhale_time * sample_rate)
        exhale_samples = int(exhale_time * sample_rate)
        
        # === INHALE SOUND ===
        # Human inhale has a characteristic frequency distribution
        # Primarily around 1kHz-2kHz with some higher harmonics
        
        # Create core breath sound based on filtered noise
        noise_inhale = np.random.normal(0, 1, inhale_samples)
        
        # Multi-band filtering to simulate throat and nasal passage resonances
        # First band: ~800-1200Hz - primary airflow component
        b1, a1 = signal.butter(2, [0.036, 0.054], 'bandpass')  # ~800-1200Hz at 44.1kHz
        filtered_band1 = signal.lfilter(b1, a1, noise_inhale)
        
        # Second band: ~1500-2000Hz - nasal component
        b2, a2 = signal.butter(2, [0.068, 0.09], 'bandpass')  # ~1500-2000Hz
        filtered_band2 = signal.lfilter(b2, a2, noise_inhale)
        
        # Third band: ~400-700Hz - throat component
        b3, a3 = signal.butter(2, [0.018, 0.032], 'bandpass')  # ~400-700Hz
        filtered_band3 = signal.lfilter(b3, a3, noise_inhale)
        
        # Mix the bands with specific amplitudes
        inhale_sound = (filtered_band1 * 0.7 + 
                       filtered_band2 * 0.4 + 
                       filtered_band3 * 0.45)
        
        # Dynamic amplitude envelope for inhale
        # Starts slow, accelerates in the middle, then levels off
        t = np.linspace(0, 1, inhale_samples)
        inhale_env = 0.1 + 0.9 * (1 - np.cos(t * np.pi)) / 2  # Cosine curve for natural acceleration
        
        # Apply some subtle "catches" to simulate natural breath
        # Small random variations in amplitude
        micro_variations = 1.0 + 0.05 * np.sin(t * 2 * np.pi * 8)  # 8 subtle variations
        inhale_env = inhale_env * micro_variations
        
        # Apply envelope
        inhale_sound = inhale_sound * inhale_env
        
        # Smooth attack
        attack_samples = int(0.08 * inhale_samples)  # 8% smooth attack
        if attack_samples > 0:
            attack_env = np.linspace(0, 1, attack_samples)
            inhale_sound[:attack_samples] *= attack_env
        
        # === EXHALE SOUND ===
        # Human exhale has more low frequency content compared to inhale
        # More resonance in 300-800Hz range with some whistle-like higher frequencies
        
        noise_exhale = np.random.normal(0, 1, exhale_samples)
        
        # First band: ~300-600Hz - primary exhale component (stronger than inhale)
        b1, a1 = signal.butter(2, [0.014, 0.027], 'bandpass')  # ~300-600Hz
        filtered_band1 = signal.lfilter(b1, a1, noise_exhale)
        
        # Second band: ~600-1000Hz - mouth/lips component
        b2, a2 = signal.butter(2, [0.027, 0.045], 'bandpass')  # ~600-1000Hz
        filtered_band2 = signal.lfilter(b2, a2, noise_exhale)
        
        # Third band: ~1500-2500Hz - slight "whistle" component
        b3, a3 = signal.butter(2, [0.068, 0.113], 'bandpass')  # ~1500-2500Hz
        filtered_band3 = signal.lfilter(b3, a3, noise_exhale)
        
        # Mix the bands - exhale has stronger low frequencies
        exhale_sound = (filtered_band1 * 0.8 + 
                       filtered_band2 * 0.45 + 
                       filtered_band3 * 0.25)
        
        # Dynamic amplitude envelope for exhale
        # Initial push followed by gradual decay
        t = np.linspace(0, 1, exhale_samples)
        exhale_env = 0.2 + 0.8 * np.exp(-t * 2.5)  # Exponential decay
        
        # Add subtle "pushes" to simulate natural exhale
        micro_variations = 1.0 + 0.03 * np.sin(t * 2 * np.pi * 6)  # 6 subtle variations
        exhale_env = exhale_env * micro_variations
        
        # Apply envelope
        exhale_sound = exhale_sound * exhale_env
        
        # Smooth release
        release_samples = int(0.15 * exhale_samples)  # 15% smooth release
        if release_samples > 0:
            release_env = np.linspace(1, 0, release_samples)
            exhale_sound[-release_samples:] *= release_env
        
        # Final scale to avoid clipping
        inhale_max = np.max(np.abs(inhale_sound))
        if inhale_max > 0:
            inhale_sound = inhale_sound / inhale_max * 0.85
            
        exhale_max = np.max(np.abs(exhale_sound))
        if exhale_max > 0:
            exhale_sound = exhale_sound / exhale_max * 0.9
        
        return inhale_sound, exhale_sound
    
    # Generate full session
    cycles = int(duration / (inhale_time + exhale_time))
    print(f"Generating meditation breathing for {duration} seconds ({cycles} cycles)")
    
    # Pre-allocate full audio array
    full_audio = np.array([], dtype=np.float32)
    
    # Generate each cycle
    for i in range(cycles):
        print(f"Generating cycle {i+1}/{cycles}")
        
        # Get inhale and exhale sounds for this cycle
        inhale_sound, exhale_sound = generate_human_breath_cycle()
        
        # Play audio if requested
        if play_audio:
            print("Inhale...")
            sd.play(inhale_sound, sample_rate)
            sd.wait()
            
            print("Exhale...")
            sd.play(exhale_sound, sample_rate)
            sd.wait()
        
        # Append to full audio
        full_audio = np.append(full_audio, inhale_sound)
        full_audio = np.append(full_audio, exhale_sound)
    
    # Insert a brief pause between cycles
    pause_length = int(0.1 * sample_rate)  # 0.1 seconds
    full_audio_with_pauses = np.array([], dtype=np.float32)
    
    cycle_length = int((inhale_time + exhale_time) * sample_rate)
    for i in range(0, len(full_audio), cycle_length):
        if i + cycle_length <= len(full_audio):
            full_audio_with_pauses = np.append(full_audio_with_pauses, full_audio[i:i+cycle_length])
            if i + cycle_length < len(full_audio):  # Don't add pause after the last cycle
                full_audio_with_pauses = np.append(full_audio_with_pauses, np.zeros(pause_length))
    
    full_audio = full_audio_with_pauses
    print(f"Generated audio length: {len(full_audio)/sample_rate:.2f} seconds")
    
    # Save as WAV file
    if save_wav:
        wav_filename = f"{output_filename}.wav"
        print(f"Saving to {wav_filename}...")
        
        # Normalize to avoid clipping
        max_amplitude = np.max(np.abs(full_audio))
        if max_amplitude > 0:
            full_audio = full_audio / max_amplitude * 0.95
            
        # Convert to 16-bit int format
        normalized_audio = np.int16(full_audio * 32767)
        wavfile.write(wav_filename, sample_rate, normalized_audio)
        print(f"WAV file saved")
    
    # Save as MP3 file
    if save_mp3 and PYDUB_AVAILABLE:
        mp3_filename = f"{output_filename}.mp3"
        print(f"Converting to MP3 format ({mp3_filename})...")
        
        # Create a temporary WAV for conversion
        temp_wav = "_temp_for_mp3_conversion.wav"
        # Normalize to avoid clipping
        max_amplitude = np.max(np.abs(full_audio))
        if max_amplitude > 0:
            full_audio = full_audio / max_amplitude * 0.95
        normalized_audio = np.int16(full_audio * 32767)
        wavfile.write(temp_wav, sample_rate, normalized_audio)
        
        # Convert WAV to MP3
        try:
            audio = AudioSegment.from_wav(temp_wav)
            audio.export(mp3_filename, format="mp3", bitrate="192k")
            print(f"MP3 file saved successfully")
            
            # Remove temp file
            import os
            if os.path.exists(temp_wav):
                os.remove(temp_wav)
        except Exception as e:
            print(f"Error converting to MP3: {e}")
    
    print("Meditation audio generation complete.")
    return full_audio

# Run the script
if __name__ == "__main__":
    audio = generate_meditation_sounds(
        inhale_time=6, 
        exhale_time=6, 
        duration=24,  # 2 minutes
        play_audio=True,
        save_wav=True,
        save_mp3=False,
        output_filename="human_breathing_meditation"
    )

Generating meditation breathing for 24 seconds (2 cycles)
Generating cycle 1/2
Inhale...
Exhale...
Generating cycle 2/2
Inhale...
Exhale...


KeyboardInterrupt: 